# edl-ml walkthrough

End-to-end demo of the `edl-ml` package: physics simulation, dataset generation, surrogate training, and interpretability. Runs in a few minutes on a laptop CPU.

## 1. Physics — Poisson-Boltzmann and Gouy-Chapman-Stern

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from edl_ml.physics import (
    PBParameters, solve_poisson_boltzmann, debye_length,
    GCSParameters, gouy_chapman_stern, total_capacitance,
)
from edl_ml.viz import plot_ion_profiles, plot_capacitance_curve

print(f'Debye length @ 0.1 M : {debye_length(0.1, 1)*1e9:.3f} nm')
print(f'Debye length @ 1 mM  : {debye_length(1e-3, 1)*1e9:.3f} nm')

res = solve_poisson_boltzmann(PBParameters(0.01, psi_diffuse_v=0.1))
plot_ion_profiles(res, title='c=10 mM, ψ_d=100 mV');

In [ ]:
params = GCSParameters(concentration_mol_l=0.1)
E = np.linspace(-0.4, 0.4, 81)
sigma, psi_d, cap = gouy_chapman_stern(params, E)
plot_capacitance_curve(E, cap * 100.0, title='GCS capacitance at 0.1 M');

## 2. Dataset — Latin hypercube over the five physical parameters

In [ ]:
from edl_ml.data import SamplingBounds, build_capacitance_dataset, split_by_sample

df = build_capacitance_dataset(SamplingBounds(potential_n_points=51), n_samples=400, seed=0)
print(df.head())
train, val, test = split_by_sample(df)
print('sizes:', len(train), len(val), len(test))

## 3. Train the surrogate

In [ ]:
from edl_ml.ml import MLPConfig, TrainConfig, build_loaders, train_model
from edl_ml.ml.dataset import INPUT_COLUMNS
from edl_ml.viz import plot_loss_curves, plot_parity, plot_error_distribution

loaders = build_loaders(train, val, test, batch_size=256)
report = train_model(
    loaders,
    MLPConfig(input_dim=len(INPUT_COLUMNS), hidden_dims=(128, 128, 64), dropout=0.05),
    TrainConfig(max_epochs=150, learning_rate=1e-3, patience=30),
)
print('test metrics:', report.test_metrics)
plot_loss_curves(report.train_losses, report.val_losses, title='Training curves');

## 4. Parity and residual diagnostics

In [ ]:
import torch
report.model.eval()
preds, trues = [], []
with torch.no_grad():
    for x, y in loaders.test:
        p = loaders.y_scaler.inverse_transform(report.model(x).cpu()).numpy()
        t = loaders.y_scaler.inverse_transform(y).numpy()
        preds.append(p); trues.append(t)
y_pred = np.concatenate(preds).ravel()
y_true = np.concatenate(trues).ravel()
plot_parity(y_true, y_pred, title='Test parity (µF/cm²)');
plot_error_distribution(y_true, y_pred, title='Test residuals (µF/cm²)');

## 5. Interpretability — permutation importance

In [ ]:
from edl_ml.ml.explain import permutation_feature_importance
feats = test[list(INPUT_COLUMNS)].to_numpy(dtype=np.float32)
targets = test['capacitance_uf_cm2'].to_numpy(dtype=np.float32)
imp = permutation_feature_importance(report.model, feats, targets, loaders.x_scaler, loaders.y_scaler, n_repeats=10)
for name, value in zip(INPUT_COLUMNS, imp):
    print(f'{name:30s} ΔMAE = {value:+.4f} µF/cm²')